# NumPy - Exercise Solutions

In [1]:
import numpy as np

### a)

* You have a 3D MRI scan of a single subject (100x100x100 voxels). Due to "bias field" effects, different slices (along the Z-axis) have slightly different average intensities. To correct this, you need to calculate the mean intensity for each 2D slice and subtract that mean from every voxel in that specific slice.

* The task:
  * Create a 100x100x100 array of random floats to simulate the MRI volume.
  * Calculate the mean intensity for each of the 100 slices along the Z (third) axis (axis=2).
  * Reshape the resulting means and use Broadcasting to normalize the entire volume.
    * Hint: Reshape the 1D means into a 3D shape that allows NumPy to 'stretch' each mean value across all voxels in its corresponding slice before performing the subtraction.
  * Raise your hand and explain how you checked the correctness of your code.
    * I'm interested in hearing you explain how you know the mean was computed on the right slices, and how the right reshaping/bradcasting was done.  
    * Hint: Since the goal is to subtract the mean from every slice, the new mean of every slice should be (theoretically) zero. And the global Standard deviation should stay the same.

In [2]:
# Create a 3D "brain" volume (100x100x100)
# We use random.normal to simulate realistic intensity variations - just one possible way of doing it!
mri_volume = np.random.normal(loc=120, scale=10, size=(100, 100, 100))


# Check the initial shape and size
print(f"Volume Shape: {mri_volume.shape}") # (100, 100, 100)
print(f"Total Voxels: {mri_volume.size}")  # 1,000,000
print(f"Mean of first slice before normalization: {mri_volume[:,:,0].mean():.4f}")
print(f"Global Std before: {mri_volume.std():.4f}")

# Calculate the mean for each 2D slice along the Z-axis (axis=2)
# This results in an array of 100 mean values.
slice_means = mri_volume.mean(axis=(0, 1)) 

# Reshape the means to (1, 1, 100) so they can broadcast across 
# the (100, 100, 100) volume.
slice_means_reshaped = slice_means.reshape((1, 1, 100))

# Subtract the means using vectorization
# This happens at near-C speeds despite 1 million calculations!
normalized_volume = mri_volume - slice_means_reshaped

print("\nNormalization Complete.")
print(f"Mean of first slice after normalization: {normalized_volume[:,:,0].mean():.4f}")
print(f"Global Std after:  {normalized_volume.std():.4f}")

Volume Shape: (100, 100, 100)
Total Voxels: 1000000
Mean of first slice before normalization: 119.9351
Global Std before: 10.0072

Normalization Complete.
Mean of first slice after normalization: 0.0000
Global Std after:  10.0067


# Pandas - Exercise Solutions

In [3]:
import pandas as pd  # Importing the pandas module

### a)
* Read in the `Lectures/2026/data/participants_nbsub-100.tsv` file using Pandas.
  The difference between a CSV and a TSV is that the separator is either a comma (C) or a tab (T),
  and this separator can be specified as an argument to the read_csv method.
* Only keep the columns 'SUB_ID', 'SITE_ID', 'FILE_ID', 'AGE_AT_SCAN', and 'SEX'
* Find the dimensionality of the dataframe, and display the first 20 rows.

In [4]:
csv_data = pd.read_csv(
    "participants_nbsub-200.tsv",
    sep="\t",
    usecols=["SUB_ID", "SITE_ID", "FILE_ID", "AGE_AT_SCAN", "SEX"],
)
print(csv_data.shape)
display(csv_data.head(20))

(200, 5)


,SUB_ID,SITE_ID,FILE_ID,AGE_AT_SCAN,SEX
0,50003,PITT,Pitt_0050003,24.45,1
1,50004,PITT,Pitt_0050004,19.09,1
2,50005,PITT,Pitt_0050005,13.73,2
3,50006,PITT,Pitt_0050006,13.37,1
4,50007,PITT,Pitt_0050007,17.78,1
5,50008,PITT,Pitt_0050008,32.45,1
6,50010,PITT,Pitt_0050010,35.20,1
7,50011,PITT,Pitt_0050011,16.93,1
8,50012,PITT,Pitt_0050012,21.48,1
9,50013,PITT,Pitt_0050013,9.33,1


### b) 
* Make a new dataframe containing all of the subjects below age 15, that are NOT found at the 'PITT' SITE_ID, and that are males.
* Find the dimensionality of this resulting dataframe, how did it change? Look at the order of the indices, how did they change?
* Use the describe() method to find a statistical summary of the resulting AGE_AT_SCAN Series.
* Reset the index of the dataframe as seen in the [Documentation](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.reset_index.html), such that the dataframe indices are re-set to start from 0.

In [5]:
df_query = csv_data.loc[
    (csv_data["SITE_ID"] != "PITT") & (csv_data["AGE_AT_SCAN"] < 15.0) & (csv_data["SEX"] == 1)
]  # Note you could also find the indices of such occurrences, and drop them.
print(df_query.shape)
print(df_query["AGE_AT_SCAN"].describe())
df_query = df_query.reset_index(drop=True)
display(df_query.head(20))

(72, 5)
count    72.000000
mean     12.199306
std       1.867100
min       8.000000
25%      10.612500
50%      12.490000
75%      13.902500
max      14.910000
Name: AGE_AT_SCAN, dtype: float64


,SUB_ID,SITE_ID,FILE_ID,AGE_AT_SCAN,SEX
0,50102,OLIN,Olin_0050102,14.00,1
1,50103,OLIN,Olin_0050103,14.00,1
2,50106,OLIN,Olin_0050106,10.00,1
3,50111,OLIN,Olin_0050111,14.00,1
4,50129,OLIN,Olin_0050129,12.00,1
5,50135,OLIN,Olin_0050135,12.00,1
6,50142,OHSU,OHSU_0050142,13.99,1
7,50143,OHSU,OHSU_0050143,13.79,1
8,50144,OHSU,OHSU_0050144,10.22,1
9,50145,OHSU,OHSU_0050145,10.75,1


### c)
* Say you are given three Series containing the SUB_ID, the weights and the height of subjects.
* Make a dataframe out of these three Series.
* Merge the new dataframe and the existing one on based on the common SUB_ID.

In [6]:
weight_array = np.random.normal(50.0, 5.0, df_query.shape[0])
height_array = np.random.normal(160.0, 10.0, df_query.shape[0])

weight_series = pd.Series(weight_array)
height_series = pd.Series(height_array)
sub_id_series = df_query["SUB_ID"].copy()  # This df_query dataframe is the output of exercise b).

In [7]:
new_df = pd.DataFrame({"SUB_ID": sub_id_series, "weight": weight_series, "height": height_series})
display(new_df)
merged = pd.merge(df_query, new_df, on=["SUB_ID"], how="inner")
display(merged.head(20))

,SUB_ID,weight,height
0,50102,52.800894,169.369665
1,50103,47.344722,158.398267
2,50106,49.645236,154.490204
3,50111,48.556372,148.618281
4,50129,46.192320,158.623940
...,...,...,...
67,50312,50.927235,147.719211
68,50315,49.182622,159.081579
69,50318,50.316784,171.837202
70,50324,49.361574,158.427915


,SUB_ID,SITE_ID,FILE_ID,AGE_AT_SCAN,SEX,weight,height
0,50102,OLIN,Olin_0050102,14.00,1,52.800894,169.369665
1,50103,OLIN,Olin_0050103,14.00,1,47.344722,158.398267
2,50106,OLIN,Olin_0050106,10.00,1,49.645236,154.490204
3,50111,OLIN,Olin_0050111,14.00,1,48.556372,148.618281
4,50129,OLIN,Olin_0050129,12.00,1,46.192320,158.623940
5,50135,OLIN,Olin_0050135,12.00,1,44.858507,154.972985
6,50142,OHSU,OHSU_0050142,13.99,1,53.822977,156.280487
7,50143,OHSU,OHSU_0050143,13.79,1,62.064291,166.394095
8,50144,OHSU,OHSU_0050144,10.22,1,50.127154,158.050192
9,50145,OHSU,OHSU_0050145,10.75,1,46.423663,149.746904
